In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings  
import sys
sys.path.append('/home/ubuntu/da_hf1_analytics/scripts/ate_strecke')

#from hypotest import *
# Runtime-Warnungen ausschalten  
warnings.filterwarnings("ignore", category=RuntimeWarning) 
def AggregateLeads(data,av,ID):
    dat = data.groupby(ID)[av].max().reset_index()
    return dat

class DataProcessor:
    def __init__(self, data):
        self.data = data
        self.log = []
    def load_raw_data(self,path):
        self.raw_data = pd.read_sas(path)
        
    def log_data_step(self, step_name, data_before, data_after):
        count_before = len(data_before)
        count_after = len(data_after)
        lost_count = count_before - count_after
        self.log.append({
            'Step': step_name,
            'Count Before': count_before,
            'Count After': count_after,
            'Lost Count': lost_count
        })

    def apply_condition(self, condition_func, step_name="Processing"):
        """
        Apply a condition function to filter the data.

        :param condition_func: A function that takes the DataFrame and returns a boolean mask.
        :param step_name: Name of the processing step for logging.
        """
        data_before = self.data
        self.data = self.data[condition_func(self.data)]
        self.log_data_step(step_name, data_before, self.data)

    def get_log(self):
        return pd.DataFrame(self.log)

    def preprocess_target_to_flg(self,
                                 targets = ['TA_KFZ_ABSCHLUSS_CNT',
                                 'TA_HUS_ABSCHLUSS_CNT',
                                 'TA_RS_ABSCHLUSS_CNT',
                                 'TA_LV_ABSCHLUSS_CNT',
                                 'TA_KV_ABSCHLUSS_CNT',
                                 'TA_VRS_ABSCHLUSS_CNT',
                                 'TA_PBV_ABSCHLUSS_CNT',
                                 'TA_SEN_ABSCHLUSS_CNT',
                                 'TA_RSON_ABSCHLUSS_CNT',
                                 'TA_PH_ABSCHLUSS_CNT',
                                 'TA_HR_ABSCHLUSS_CNT',
                                 'TA_WG_ABSCHLUSS_CNT',
                                 'TA_UN_ABSCHLUSS_CNT',
                                 'TA_TH_ABSCHLUSS_CNT',
                                 'TA_GL_ABSCHLUSS_CNT']
                                ):
        # Create FLG features for targets
        for t in targets:
            self.data['%sFLG'%t[:-3]] = (self.data[t]>0).astype('int');

In [2]:
data_raw = pd.read_sas("/domino/edv/pvc-hf1kundehuk/LEAD_Dataprep/output/lead_husrs_tag1_90_prd_het.sas7bdat", encoding='UTF-8')

In [3]:
analyse_zeitraum = 90

data_object = DataProcessor(data_raw)
data_object.preprocess_target_to_flg();
data_object.apply_condition(lambda df: df['LEAD_ERF_ORGA_VW']  == '3 KBC', step_name=f"Selektiere Leads, die im KBC erstellt wurden")
data_object.apply_condition(lambda df: df['LEAD_STATUS_EN'] != -1 , step_name=f"Selektiere Leads, die nicht sofort beendet wurden")
data_object.apply_condition(lambda df: df['BASIS_DT'] <= df['BASIS_DT'].max()-pd.Timedelta(days=analyse_zeitraum), step_name=f"Selektiere Leads, die vor mind. {analyse_zeitraum} Tagen erstellt wurden.")
data_object.apply_condition(lambda df: df['BASIS_DT'] >= '2024-11-01', step_name=f"Selektiere Leads, die nach dem 1.11.2024 erstellt wurden")
data_object.apply_condition(lambda df: ~((df['BASIS_DT'] >= '2025-10-01') & (df['LEAD_DA_GRUPPE_TXT'] == 'Delayed - 8/10 Tag')), step_name=f"Droppe Leads, die nach dem 1.10.2025 erstellt wurden und in der Delayed-Gruppe waren")

df = data_object.data
df['Treatment'] = df.LEAD_DA_GRUPPE_TXT != 'Control'

df['AUSSPIELUNG_DT'] = np.where(df['LEAD_SENDEWEG'] == 'P',
                             df['BASIS_DT'] + pd.Timedelta(days=6),
                             df['BASIS_DT'] + pd.Timedelta(days=4))

df['TA_HUS_RS_ABSCHLUSS_CNT'] = df[['TA_HUS_ABSCHLUSS_CNT', 'TA_RS_ABSCHLUSS_CNT']].sum(axis=1)

/tmp/ipykernel_658689/1336123220.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.data['%sFLG'%t[:-3]] = (self.data[t]>0).astype('int');
/tmp/ipykernel_658689/1336123220.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  self.data['%sFLG'%t[:-3]] = (self.data[t]>0).astype('int');
/tmp/ipykernel_658689/1336123220.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(ax

In [4]:
data_object.get_log()

,Step,Count Before,Count After,Lost Count
0,"Selektiere Leads, die im KBC erstellt wurden",334372,334372,0
1,"Selektiere Leads, die nicht sofort beendet wurden",334372,334372,0
2,"Selektiere Leads, die vor mind. 90 Tagen erste...",334372,292373,41999
3,"Selektiere Leads, die nach dem 1.11.2024 erste...",292373,292373,0
4,"Droppe Leads, die nach dem 1.10.2025 erstellt ...",292373,258750,33623


In [5]:
cols_to_drop = [
 'BASIS_TAGE_DIFF',
 'BASIS_JOIN_KRIT',
 'TA_JOIN_KRIT',
 'TA_DT_JUENGST',
 'TA_DT_AELTEST',
 'TA_DT_AELTEST_KFZ',
 'TA_DT_AELTEST_HUS',
 'TA_DT_AELTEST_RS',
 'TA_DT_AELTEST_LV',
 'TA_DT_AELTEST_KV',
 'TA_DT_AELTEST_PH',
 'TA_DT_AELTEST_HR',
 'TA_DT_AELTEST_WG',
 'TA_DT_AELTEST_UN',
 'TA_DT_AELTEST_TH',
 'TA_DT_AELTEST_GL',
 'TA_DT_AELTEST_FD',
 'TA_DT_AELTEST_HUG',
 'TA_DT_AELTEST_BH',
 'TA_KFZ_ABSCHLUSS_CNT',
 'TA_HUS_ABSCHLUSS_CNT',
 'TA_RS_ABSCHLUSS_CNT',
 'TA_LV_ABSCHLUSS_CNT',
 'TA_KV_ABSCHLUSS_CNT',
 'TA_VRS_ABSCHLUSS_CNT',
 'TA_PBV_ABSCHLUSS_CNT',
 'TA_SEN_ABSCHLUSS_CNT',
 'TA_RSON_ABSCHLUSS_CNT',
 'TA_PH_ABSCHLUSS_CNT',
 'TA_HR_ABSCHLUSS_CNT',
 'TA_WG_ABSCHLUSS_CNT',
 'TA_UN_ABSCHLUSS_CNT',
 'TA_TH_ABSCHLUSS_CNT',
 'TA_GL_ABSCHLUSS_CNT',
 'TA_FD_ABSCHLUSS_CNT',
 'TA_HUG_ABSCHLUSS_CNT',
 'TA_BH_ABSCHLUSS_CNT',
 'TA_DT_JUENGST_GS',
 'TA_DT_AELTEST_GS',
 'TA_DT_AELTEST_KFZ_GS',
 'TA_DT_AELTEST_HUS_GS',
 'TA_DT_AELTEST_RS_GS',
 'TA_DT_AELTEST_LV_GS',
 'TA_DT_AELTEST_KV_GS',
 'TA_DT_AELTEST_PH_GS',
 'TA_DT_AELTEST_HR_GS',
 'TA_DT_AELTEST_WG_GS',
 'TA_DT_AELTEST_UN_GS',
 'TA_DT_AELTEST_TH_GS',
 'TA_DT_AELTEST_GL_GS',
 'TA_DT_AELTEST_FD_GS',
 'TA_DT_AELTEST_HUG_GS',
 'TA_DT_AELTEST_BH_GS',
 'TA_KFZ_ABSCHLUSS_CNT_GS',
 'TA_HUS_ABSCHLUSS_CNT_GS',
 'TA_RS_ABSCHLUSS_CNT_GS',
 'TA_LV_ABSCHLUSS_CNT_GS',
 'TA_KV_ABSCHLUSS_CNT_GS',
 'TA_PH_ABSCHLUSS_CNT_GS',
 'TA_HR_ABSCHLUSS_CNT_GS',
 'TA_WG_ABSCHLUSS_CNT_GS',
 'TA_UN_ABSCHLUSS_CNT_GS',
 'TA_TH_ABSCHLUSS_CNT_GS',
 'TA_GL_ABSCHLUSS_CNT_GS',
 'TA_FD_ABSCHLUSS_CNT_GS',
 'TA_HUG_ABSCHLUSS_CNT_GS',
 'TA_BH_ABSCHLUSS_CNT_GS',
 'TA_DT_JUENGST_KD',
 'TA_DT_AELTEST_KD',
 'TA_DT_AELTEST_KFZ_KD',
 'TA_DT_AELTEST_HUS_KD',
 'TA_DT_AELTEST_RS_KD',
 'TA_DT_AELTEST_LV_KD',
 'TA_DT_AELTEST_KV_KD',
 'TA_DT_AELTEST_PH_KD',
 'TA_DT_AELTEST_HR_KD',
 'TA_DT_AELTEST_WG_KD',
 'TA_DT_AELTEST_UN_KD',
 'TA_DT_AELTEST_TH_KD',
 'TA_DT_AELTEST_GL_KD',
 'TA_DT_AELTEST_FD_KD',
 'TA_DT_AELTEST_HUG_KD',
 'TA_DT_AELTEST_BH_KD',
 'TA_KFZ_ABSCHLUSS_CNT_KD',
 'TA_HUS_ABSCHLUSS_CNT_KD',
 'TA_RS_ABSCHLUSS_CNT_KD',
 'TA_LV_ABSCHLUSS_CNT_KD',
 'TA_KV_ABSCHLUSS_CNT_KD',
 'TA_PH_ABSCHLUSS_CNT_KD',
 'TA_HR_ABSCHLUSS_CNT_KD',
 'TA_WG_ABSCHLUSS_CNT_KD',
 'TA_UN_ABSCHLUSS_CNT_KD',
 'TA_TH_ABSCHLUSS_CNT_KD',
 'TA_GL_ABSCHLUSS_CNT_KD',
 'TA_FD_ABSCHLUSS_CNT_KD',
 'TA_HUG_ABSCHLUSS_CNT_KD',
 'TA_BH_ABSCHLUSS_CNT_KD',
 'TA_DT_JUENGST_VM',
 'TA_DT_AELTEST_VM',
 'TA_DT_AELTEST_KFZ_VM',
 'TA_DT_AELTEST_HUS_VM',
 'TA_DT_AELTEST_RS_VM',
 'TA_DT_AELTEST_LV_VM',
 'TA_DT_AELTEST_KV_VM',
 'TA_DT_AELTEST_PH_VM',
 'TA_DT_AELTEST_HR_VM',
 'TA_DT_AELTEST_WG_VM',
 'TA_DT_AELTEST_UN_VM',
 'TA_DT_AELTEST_TH_VM',
 'TA_DT_AELTEST_GL_VM',
 'TA_DT_AELTEST_FD_VM',
 'TA_DT_AELTEST_HUG_VM',
 'TA_DT_AELTEST_BH_VM',
 'TA_KFZ_ABSCHLUSS_CNT_VM',
 'TA_HUS_ABSCHLUSS_CNT_VM',
 'TA_RS_ABSCHLUSS_CNT_VM',
 'TA_LV_ABSCHLUSS_CNT_VM',
 'TA_KV_ABSCHLUSS_CNT_VM',
 'TA_PH_ABSCHLUSS_CNT_VM',
 'TA_HR_ABSCHLUSS_CNT_VM',
 'TA_WG_ABSCHLUSS_CNT_VM',
 'TA_UN_ABSCHLUSS_CNT_VM',
 'TA_TH_ABSCHLUSS_CNT_VM',
 'TA_GL_ABSCHLUSS_CNT_VM',
 'TA_FD_ABSCHLUSS_CNT_VM',
 'TA_HUG_ABSCHLUSS_CNT_VM',
 'TA_BH_ABSCHLUSS_CNT_VM',
 'TA_DT_JUENGST_OVT',
 'TA_DT_AELTEST_OVT',
 'TA_DT_AELTEST_KFZ_OVT',
 'TA_DT_AELTEST_HUS_OVT',
 'TA_DT_AELTEST_RS_OVT',
 'TA_DT_AELTEST_LV_OVT',
 'TA_DT_AELTEST_KV_OVT',
 'TA_DT_AELTEST_PH_OVT',
 'TA_DT_AELTEST_HR_OVT',
 'TA_DT_AELTEST_WG_OVT',
 'TA_DT_AELTEST_UN_OVT',
 'TA_DT_AELTEST_TH_OVT',
 'TA_DT_AELTEST_GL_OVT',
 'TA_DT_AELTEST_FD_OVT',
 'TA_DT_AELTEST_HUG_OVT',
 'TA_DT_AELTEST_BH_OVT',
 'TA_KFZ_ABSCHLUSS_CNT_OVT',
 'TA_HUS_ABSCHLUSS_CNT_OVT',
 'TA_RS_ABSCHLUSS_CNT_OVT',
 'TA_LV_ABSCHLUSS_CNT_OVT',
 'TA_KV_ABSCHLUSS_CNT_OVT',
 'TA_PH_ABSCHLUSS_CNT_OVT',
 'TA_HR_ABSCHLUSS_CNT_OVT',
 'TA_WG_ABSCHLUSS_CNT_OVT',
 'TA_UN_ABSCHLUSS_CNT_OVT',
 'TA_TH_ABSCHLUSS_CNT_OVT',
 'TA_GL_ABSCHLUSS_CNT_OVT',
 'TA_FD_ABSCHLUSS_CNT_OVT',
 'TA_HUG_ABSCHLUSS_CNT_OVT',
 'TA_BH_ABSCHLUSS_CNT_OVT',
 'TA_DT_JUENGST_KBC',
 'TA_DT_AELTEST_KBC',
 'TA_DT_AELTEST_KFZ_KBC',
 'TA_DT_AELTEST_HUS_KBC',
 'TA_DT_AELTEST_RS_KBC',
 'TA_DT_AELTEST_LV_KBC',
 'TA_DT_AELTEST_KV_KBC',
 'TA_DT_AELTEST_PH_KBC',
 'TA_DT_AELTEST_HR_KBC',
 'TA_DT_AELTEST_WG_KBC',
 'TA_DT_AELTEST_UN_KBC',
 'TA_DT_AELTEST_TH_KBC',
 'TA_DT_AELTEST_GL_KBC',
 'TA_DT_AELTEST_FD_KBC',
 'TA_DT_AELTEST_HUG_KBC',
 'TA_DT_AELTEST_BH_KBC',
 'TA_KFZ_ABSCHLUSS_CNT_KBC',
 'TA_HUS_ABSCHLUSS_CNT_KBC',
 'TA_RS_ABSCHLUSS_CNT_KBC',
 'TA_LV_ABSCHLUSS_CNT_KBC',
 'TA_KV_ABSCHLUSS_CNT_KBC',
 'TA_PH_ABSCHLUSS_CNT_KBC',
 'TA_HR_ABSCHLUSS_CNT_KBC',
 'TA_WG_ABSCHLUSS_CNT_KBC',
 'TA_UN_ABSCHLUSS_CNT_KBC',
 'TA_TH_ABSCHLUSS_CNT_KBC',
 'TA_GL_ABSCHLUSS_CNT_KBC',
 'TA_FD_ABSCHLUSS_CNT_KBC',
 'TA_HUG_ABSCHLUSS_CNT_KBC',
 'TA_BH_ABSCHLUSS_CNT_KBC',
 'TA_ABSCHLUSS_FLG',
 'TA_ABSCHLUSS_CNT',
 'TA_ABSCHLUSS_FLG_OVT',
 'TA_ABSCHLUSS_CNT_OVT',
 'TA_FOLGEANBINDUNG_FLG',
 'TA_FOLGEANBINDUNG_CNT',
 'TA_FOLGEANBINDUNG_FLG_OVT',
 'TA_FOLGEANBINDUNG_CNT_OVT',
 'PID_LAST_CHAR',
 'LEAD_ID_SEQ',
    
 'LEAD_DA_GRUPPE',
 'LEAD_DA_GRUPPE_TXT',
 
 'LEAD_GEVO_ID',
 'LEAD_ORDBEG_TYP',
 'LEAD_ORDBEG_PRUEFBUCHSTABE',
 'LEAD_ORDBEG_SPARTE_NR',

 'LEAD_ART_DETAIL',    
 'LEAD_STATUS_EN',
 'LEAD_STATUS_EN_TXT',
 'LEAD_AUSGESPIELT_FLG',
 'LEAD_ID',
 'LEAD_ID_DT',
 'LEAD_VERSIONEN_CNT',
 'LEAD_LEADOWNER',
 'LEAD_ZUSTELLKRIT',
 'LEAD_VERSION_ID',
 'LEAD_VERSION_DT',
 'LEAD_VERSION_TS',
 'LEAD_VERSION_DT_LAG',
 'LEAD_DAYDIFF_TO_LAG',
 'LEAD_DAYDIFF_TO_FIRST',
 'LEAD_DAYDIFF_1_TO_2',
 'LEAD_VERS1_SCORE',
 'LEAD_VERS2_SCORE',
 'LEAD_ERF_ORGA',
 'LEAD_ERF_ORGA_VW',
 'LEAD_BEARBEITER_ORGA',
 'LEAD_BEARBEITER_VW',
 'LEAD_BEARBEITER_GS_KDB_VM',
 'LEAD_BRB_ORGA',
 'LEAD_BRB_ORGA_VW',
 'LEAD_BRB_ORGA_GS_KDB_VM',
 'LEAD_BRB_KDB_TESTFELD2_FLG',
 'LEAD_BRB_KDB_FLG',
 'LEAD_BRB_VM_FLG',
 'LEAD_BRB_GS_FLG',
 'LEAD_BRBZU_ORGA',
 'LEAD_BRBZU_ORGA_VW',
 'LEAD_BRBZU_ORGA_GS_KDB_VM',
 'LEAD_BRBZU_KDB_TESTFELD2_FLG',
 'LEAD_BRBZU_KDB_FLG',
 'LEAD_BRBZU_VM_FLG',
 'LEAD_BRBZU_GS_FLG',
 'LEAD_NEXT_ORGA',
 'LEAD_NEXT_ORGA_VW',
 'VMKDB_VERMITTLER_NR',
 'VMKDB_VERMITTLER_NR_V',
 'VMKDB_VERMITTLER_TYP_FS',
 'VMKDB_GS_BEREICH_FS',
 'VMKDB_GS_BEREICH',
 'LEAD_AEND_GRD',
 'LEAD_AEND_GRD_TXT',
 'LEAD_AEND_GRD_EBENE_1',
 'LEAD_AEND_608_FLG',
 'LEAD_WL_FLG',
 'LEAD_NE_FLG',
 'LEAD_SLA',
 'VORC_PROD_VERC',
 'BASIS_ENDE_DT',
 'DATAPREP_LAUF_ENDE_DT',
 'DATAPREP_LAUF_TAGE_DIFF',
 'BASIS_SPARTE_GRUPPE_FS',
 'FEATURES_DATASET',
 'FEATURES_BASIS_TAGE_DIFF']

In [6]:
df = df.drop(columns=cols_to_drop, errors='ignore')
df

,PARTNER_ID_V,PARTNER_ID,LEAD_ID_STRING,LEAD_ART,LEAD_ART_DETAIL_TXT,LEAD_SENDEWEG,WVRM_WUNSCHVM_FLG,BASIS_DT,TA_DT_JUENGST_KDB,TA_DT_AELTEST_KDB,...,TA_RSON_ABSCHLUSS_FLG,TA_PH_ABSCHLUSS_FLG,TA_HR_ABSCHLUSS_FLG,TA_WG_ABSCHLUSS_FLG,TA_UN_ABSCHLUSS_FLG,TA_TH_ABSCHLUSS_FLG,TA_GL_ABSCHLUSS_FLG,Treatment,AUSSPIELUNG_DT,TA_HUS_RS_ABSCHLUSS_CNT
0,475.0,4266.0,24JUL2026:10:17:50.762692,RS_NF,Angebotsnachfass Rechtsschutz,P,0.0,2026-07-24,NaT,NaT,...,0,0,0,0,0,0,0,True,2026-07-30,0.0
1,715.0,6520.0,22DEC2025:16:43:17.831162,RS_NF,Angebotsnachfass Rechtsschutz,E,0.0,2025-12-22,NaT,NaT,...,0,0,0,0,0,0,0,False,2025-12-26,1.0
2,715.0,6520.0,22DEC2025:16:40:01.164467,HS_NF,Angebotsnachfass HUS - Privathaftpflicht,E,0.0,2025-12-22,NaT,NaT,...,0,1,0,0,0,0,0,False,2025-12-26,1.0
3,715.0,6520.0,22DEC2025:16:35:53.384735,HS_NF,Angebotsnachfass HUS - Hausrat mit Fahrradschutz,E,0.0,2025-12-22,NaT,NaT,...,0,0,1,0,0,0,0,False,2025-12-26,1.0
4,1071.0,10120.0,25NOV2025:16:34:36.242665,RS_NF,Angebotsnachfass Rechtsschutz,E,0.0,2025-11-25,NaT,NaT,...,0,0,0,0,0,0,0,False,2025-11-29,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
329370,152128412.0,81016367.0,31JUL2026:12:11:24.004657,RS_NF,Angebotsnachfass Rechtsschutz,E,0.0,2026-07-31,NaT,NaT,...,0,0,0,0,0,0,0,True,2026-08-04,0.0
329371,152128412.0,81016367.0,31JUL2026:12:12:12.695667,RS_NF,Angebotsnachfass Rechtsschutz,E,0.0,2026-07-31,NaT,NaT,...,0,0,0,0,0,0,0,True,2026-08-04,0.0
329372,152128412.0,81016367.0,31JUL2026:12:13:33.527843,HS_NF,Angebotsnachfass HUS - Privathaftpflicht,E,0.0,2026-07-31,NaT,NaT,...,0,0,0,0,0,0,0,True,2026-08-04,0.0
329373,152128412.0,81016367.0,31JUL2026:12:16:09.359525,HS_NF,Angebotsnachfass HUS - Hausrat,E,0.0,2026-07-31,NaT,NaT,...,0,0,0,0,0,0,0,True,2026-08-04,0.0


In [7]:
from datetime import datetime

def get_current_datetime_string():
    return datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

print(get_current_datetime_string())
# Output: 2024-01-15 14:23:47

2026_10_30_14_57_04


In [8]:
df['Treatment'] = df['Treatment'].replace(False, 0)

In [9]:
df['Treatment'] = df['Treatment'].replace(True, 1)

/tmp/ipykernel_658689/1985365145.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Treatment'] = df['Treatment'].replace(True, 1)


In [10]:
df

,PARTNER_ID_V,PARTNER_ID,LEAD_ID_STRING,LEAD_ART,LEAD_ART_DETAIL_TXT,LEAD_SENDEWEG,WVRM_WUNSCHVM_FLG,BASIS_DT,TA_DT_JUENGST_KDB,TA_DT_AELTEST_KDB,...,TA_RSON_ABSCHLUSS_FLG,TA_PH_ABSCHLUSS_FLG,TA_HR_ABSCHLUSS_FLG,TA_WG_ABSCHLUSS_FLG,TA_UN_ABSCHLUSS_FLG,TA_TH_ABSCHLUSS_FLG,TA_GL_ABSCHLUSS_FLG,Treatment,AUSSPIELUNG_DT,TA_HUS_RS_ABSCHLUSS_CNT
0,475.0,4266.0,24JUL2026:10:17:50.762692,RS_NF,Angebotsnachfass Rechtsschutz,P,0.0,2026-07-24,NaT,NaT,...,0,0,0,0,0,0,0,1,2026-07-30,0.0
1,715.0,6520.0,22DEC2025:16:43:17.831162,RS_NF,Angebotsnachfass Rechtsschutz,E,0.0,2025-12-22,NaT,NaT,...,0,0,0,0,0,0,0,0,2025-12-26,1.0
2,715.0,6520.0,22DEC2025:16:40:01.164467,HS_NF,Angebotsnachfass HUS - Privathaftpflicht,E,0.0,2025-12-22,NaT,NaT,...,0,1,0,0,0,0,0,0,2025-12-26,1.0
3,715.0,6520.0,22DEC2025:16:35:53.384735,HS_NF,Angebotsnachfass HUS - Hausrat mit Fahrradschutz,E,0.0,2025-12-22,NaT,NaT,...,0,0,1,0,0,0,0,0,2025-12-26,1.0
4,1071.0,10120.0,25NOV2025:16:34:36.242665,RS_NF,Angebotsnachfass Rechtsschutz,E,0.0,2025-11-25,NaT,NaT,...,0,0,0,0,0,0,0,0,2025-11-29,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
329370,152128412.0,81016367.0,31JUL2026:12:11:24.004657,RS_NF,Angebotsnachfass Rechtsschutz,E,0.0,2026-07-31,NaT,NaT,...,0,0,0,0,0,0,0,1,2026-08-04,0.0
329371,152128412.0,81016367.0,31JUL2026:12:12:12.695667,RS_NF,Angebotsnachfass Rechtsschutz,E,0.0,2026-07-31,NaT,NaT,...,0,0,0,0,0,0,0,1,2026-08-04,0.0
329372,152128412.0,81016367.0,31JUL2026:12:13:33.527843,HS_NF,Angebotsnachfass HUS - Privathaftpflicht,E,0.0,2026-07-31,NaT,NaT,...,0,0,0,0,0,0,0,1,2026-08-04,0.0
329373,152128412.0,81016367.0,31JUL2026:12:16:09.359525,HS_NF,Angebotsnachfass HUS - Hausrat,E,0.0,2026-07-31,NaT,NaT,...,0,0,0,0,0,0,0,1,2026-08-04,0.0


In [11]:
df.BEA_SCORE_WERT_HIST.value_counts(dropna=False)

BEA_SCORE_WERT_HIST
NaN     124967
1.0      44081
2.0      17458
3.0      14380
4.0      13268
5.0      10938
6.0       9031
7.0       7641
8.0       6312
9.0       5733
10.0      4941
Name: count, dtype: int64

In [12]:
#BEA NaNs füllen
df['BEA_SCORE_WERT_HIST'] = df['BEA_SCORE_WERT_HIST'].fillna(df['BEA_SCORE_WERT_HIST'].mean())

In [13]:
df.BEA_SCORE_WERT_HIST.value_counts(dropna=False)

BEA_SCORE_WERT_HIST
3.655734     124967
1.000000      44081
2.000000      17458
3.000000      14380
4.000000      13268
5.000000      10938
6.000000       9031
7.000000       7641
8.000000       6312
9.000000       5733
10.000000      4941
Name: count, dtype: int64

In [14]:
#jeweils zufällig ein Sample pro Kunde ziehen
df['_random_score'] = np.random.default_rng().random(len(df))

idx_to_keep = df.groupby('PARTNER_ID_V')['_random_score'].idxmax()

df = df.loc[idx_to_keep].drop(columns=['_random_score'])

In [15]:
df

,PARTNER_ID_V,PARTNER_ID,LEAD_ID_STRING,LEAD_ART,LEAD_ART_DETAIL_TXT,LEAD_SENDEWEG,WVRM_WUNSCHVM_FLG,BASIS_DT,TA_DT_JUENGST_KDB,TA_DT_AELTEST_KDB,...,TA_RSON_ABSCHLUSS_FLG,TA_PH_ABSCHLUSS_FLG,TA_HR_ABSCHLUSS_FLG,TA_WG_ABSCHLUSS_FLG,TA_UN_ABSCHLUSS_FLG,TA_TH_ABSCHLUSS_FLG,TA_GL_ABSCHLUSS_FLG,Treatment,AUSSPIELUNG_DT,TA_HUS_RS_ABSCHLUSS_CNT
0,475.0,4266.0,24JUL2026:10:17:50.762692,RS_NF,Angebotsnachfass Rechtsschutz,P,0.0,2026-07-24,NaT,NaT,...,0,0,0,0,0,0,0,1,2026-07-30,0.0
1,715.0,6520.0,22DEC2025:16:43:17.831162,RS_NF,Angebotsnachfass Rechtsschutz,E,0.0,2025-12-22,NaT,NaT,...,0,0,0,0,0,0,0,0,2025-12-26,1.0
4,1071.0,10120.0,25NOV2025:16:34:36.242665,RS_NF,Angebotsnachfass Rechtsschutz,E,0.0,2025-11-25,NaT,NaT,...,0,0,0,0,0,0,0,0,2025-11-29,1.0
6,1737.0,16986.0,13NOV2025:08:48:18.521443,HS_NF,Angebotsnachfass HUS - Privathaftpflicht,P,0.0,2025-11-13,NaT,NaT,...,0,1,0,0,0,0,0,1,2025-11-19,1.0
8,1898.0,18808.0,04MAY2026:14:07:02.335061,HS_NF,Angebotsnachfass HUS - Privathaftpflicht,P,0.0,2026-05-04,NaT,NaT,...,0,0,0,0,0,0,0,1,2026-05-10,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
329355,152126062.0,81014245.0,31JUL2026:09:07:27.286778,HS_NF,Angebotsnachfass HUS - Privathaftpflicht,E,0.0,2026-07-31,NaT,NaT,...,0,0,0,0,0,0,0,1,2026-08-04,0.0
329356,152126213.0,81015229.0,31JUL2026:10:38:00.324448,HS_NF,Angebotsnachfass HUS - HuG-Haftpflicht,E,0.0,2026-07-31,NaT,NaT,...,0,0,0,0,0,0,0,1,2026-08-04,0.0
329360,152126445.0,81015511.0,31JUL2026:11:57:45.713921,HS_NF,Angebotsnachfass HUS - Wohngebäude,E,0.0,2026-07-31,NaT,NaT,...,0,0,0,0,0,0,0,0,2026-08-04,0.0
329361,152126668.0,81019488.0,31JUL2026:18:27:37.260359,HS_NF,Angebotsnachfass HUS - Privathaftpflicht,E,0.0,2026-07-31,NaT,NaT,...,0,0,0,0,0,0,0,1,2026-08-04,0.0


In [16]:
pip install 'holidays==0.102'

Looking in indexes: https://nexus.lan.huk-coburg.de/repository/pypi/simple
    yapf (>='0.28') ; python_version < "3.6"
         ~^
Note: you may need to restart the kernel to use updated packages.


In [17]:
"""
Zyklische Datums-Features für kausale Analysestrecken (X / Effekt-Heterogenität).
 
Bundesweiter Use Case: Feiertage werden nicht binär kodiert, sondern als
bevölkerungsgewichteter Anteil der Bundesländer, in denen der Tag gesetzlicher
Feiertag ist. Fronleichnam ist damit ~0.64 und nicht 0 oder 1.
 
Designprinzipien (production-safe):
  * Stateless: reine Funktion des Datums, kein fit(), kein Trainings-State.
  * Beschränkt: jedes Feature liegt in [-1, 1] bzw. [0, 1]. Kein Trend, keine
    Jahreszahl, kein Zeitindex -> keine Extrapolation ausserhalb des
    Trainingsbereichs.
  * Deterministisch und batch-invariant: eine einzelne Zeile liefert exakt
    denselben Vektor wie dieselbe Zeile im Batch.
  * Stabile Spaltenreihenfolge für Serving.
 
Abhängigkeit: holidays (Version pinnen, s. _REQUIRED_HOLIDAYS_NOTE).
"""
 
from __future__ import annotations
 
import warnings
from functools import lru_cache
from typing import Union
 
import numpy as np
import pandas as pd
 
try:
    import holidays as _holidays
except ImportError:  # pragma: no cover
    _holidays = None
 
_REQUIRED_HOLIDAYS_NOTE = (
    "pip install 'holidays==0.102' - Version pinnen. Neue Releases korrigieren "
    "Kalender teils rückwirkend; ohne Pin driften Trainings- und Serving-Features "
    "auseinander."
)
 
# Bevölkerung je Bundesland in Mio., gerundet (Destatis, Stand 31.12.2023).
# Nur als Gewicht verwendet und intern normiert - die zweite Nachkommastelle
# ist für den Feature-Wert irrelevant. Bei Bedarf durch die tatsächliche
# Bestandsverteilung (Kunden je Bundesland) ersetzen, s. `weights`.
POPULATION_WEIGHTS: dict[str, float] = {
    "NW": 18.14, "BY": 13.44, "BW": 11.28, "NI": 8.14,
    "HE": 6.39,  "RP": 4.16,  "SN": 4.09,  "BE": 3.76,
    "SH": 2.95,  "BB": 2.57,  "ST": 2.17,  "TH": 2.13,
    "HH": 1.89,  "MV": 1.63,  "SL": 0.99,  "HB": 0.70,
}
 
# Anteil der bayerischen Bevölkerung in Gemeinden mit überwiegend katholischer
# Bevölkerung -> dort ist Mariä Himmelfahrt gesetzlicher Feiertag.
CATHOLIC_SHARE_BY = 0.56
 
# Präfix aller generierten Spalten. Als Konstante, damit Downstream-Code
# (Feature-Auswahl, X/W-Split, Serving-Schema) nicht auf einen String-Literal
# angewiesen ist.
TIME_FEATURE_PREFIX = "TIME_FEAT_"
 
# Mapping auf die 16 ISO-Kürzel. Deckt ausgeschriebene Namen (mit und ohne
# Umlaute), Destatis-Regionalschlüssel und gängige Hausschreibweisen ab.
_SUBDIV_ALIASES: dict[str, str] = {
    "SCHLESWIGHOLSTEIN": "SH", "01": "SH", "1": "SH",
    "HAMBURG": "HH", "02": "HH", "2": "HH",
    "NIEDERSACHSEN": "NI", "03": "NI", "3": "NI", "NDS": "NI",
    "BREMEN": "HB", "04": "HB", "4": "HB", "FREIEHANSESTADTBREMEN": "HB",
    "NORDRHEINWESTFALEN": "NW", "05": "NW", "5": "NW", "NRW": "NW",
    "HESSEN": "HE", "06": "HE", "6": "HE",
    "RHEINLANDPFALZ": "RP", "07": "RP", "7": "RP", "RLP": "RP",
    "BADENWUERTTEMBERG": "BW", "BADENWURTTEMBERG": "BW", "08": "BW", "8": "BW",
    "BAYERN": "BY", "09": "BY", "9": "BY", "BAY": "BY", "FREISTAATBAYERN": "BY",
    "SAARLAND": "SL", "10": "SL", "SAAR": "SL",
    "BERLIN": "BE", "11": "BE",
    "BRANDENBURG": "BB", "12": "BB",
    "MECKLENBURGVORPOMMERN": "MV", "13": "MV", "MECKPOMM": "MV",
    "SACHSEN": "SN", "14": "SN", "FREISTAATSACHSEN": "SN",
    "SACHSENANHALT": "ST", "15": "ST",
    "THUERINGEN": "TH", "THURINGEN": "TH", "16": "TH",
}
 
 
def _maybe_decode(v):
    """Dekodiert bytes-Werte, wie sie pandas.read_sas() ohne encoding= liefert.
    UTF-8 zuerst, dann cp1252 (SAS WLATIN1). Nicht-bytes bleiben unberuehrt."""
    if not isinstance(v, bytes):
        return v
    for enc in ("utf-8", "cp1252"):
        try:
            return v.decode(enc)
        except UnicodeDecodeError:
            continue
    return v.decode("cp1252", errors="replace")
 
 
def normalize_subdiv(values: pd.Series) -> pd.Series:
    """
    Normalisiert Bundesland-Angaben auf die 16 ISO-Kürzel ("NW", "BY", ...).
 
    Toleriert Kleinschreibung, Leerzeichen, Bindestriche, Umlaute, ausgeschriebene
    Namen, amtliche Titelzusätze ("Freistaat Bayern", "Freie und Hansestadt
    Hamburg") und Destatis-Schlüssel ("09" -> "BY"). Nicht auflösbare Werte
    werden zu NA - sie brechen die Pipeline nicht ab, sondern laufen in den
    Fallback von make_date_features.
    """
    values = pd.Series(values)
    if values.dtype == object or str(values.dtype) == "category":
        values = values.astype(object).map(_maybe_decode)
    s = values.astype("string")
    key = (
        s.str.strip()
        .str.upper()
        .str.replace("Ä", "AE", regex=False)
        .str.replace("Ö", "OE", regex=False)
        .str.replace("Ü", "UE", regex=False)
        .str.replace("ß", "SS", regex=False)
        .str.replace(r"[^A-Z0-9]", "", regex=True)
    )
    key = key.where(key.str.len() > 0, pd.NA)
 
    codes = list(POPULATION_WEIGHTS)
 
    # Kaskade: direktes Kürzel -> Alias-Tabelle -> Alias nach Entfernen der
    # amtlichen Titelzusätze. Die dritte Stufe läuft bewusst zuletzt, damit
    # "RHEINLANDPFALZ" nicht versehentlich sein "LAND" verliert.
    resolved = key.where(key.isin(codes))
    resolved = resolved.fillna(key.map(_SUBDIV_ALIASES))
 
    stripped = key.str.replace(
        r"^(FREISTAAT|FREIEUNDHANSESTADT|FREIEHANSESTADT|HANSESTADT|LAND)+",
        "",
        regex=True,
    )
    resolved = resolved.fillna(stripped.where(stripped.isin(codes)))
    resolved = resolved.fillna(stripped.map(_SUBDIV_ALIASES))
 
    return resolved.where(resolved.isin(codes), pd.NA).astype("string")
 
 
def time_feature_columns(
    df: pd.DataFrame, prefix: str = TIME_FEATURE_PREFIX
) -> list[str]:
    """Spaltennamen der Zeitfeatures in `df`, in stabiler Reihenfolge."""
    return [c for c in df.columns if c.startswith(prefix)]
 
 
def plz_to_subdiv(
    plz: pd.Series,
    mapping: Union[dict, pd.Series, "pd.DataFrame", str],
    *,
    plz_key: str = "plz",
    subdiv_key: str = "bundesland",
) -> pd.Series:
    """
    Löst Postleitzahlen zu Bundesland-Kürzeln auf.
 
    Die Zuordnungstabelle wird bewusst nicht mitgeliefert: PLZ-Grenzen fallen
    nicht mit Gemeinde- oder Landesgrenzen zusammen, ~90 PLZ liegen in zwei
    Bundesländern. Eine geratene Prefix-Heuristik (erste zwei Stellen) liegt je
    nach Region zweistellig daneben. Nutze die hausinterne Tabelle - in einem
    Versicherungsbestand existiert sie durch die Regionalklassen-Tarifierung
    ohnehin - oder einen offiziellen Datensatz (OpenPLZ, Destatis-GV-ISys).
 
    Parameters
    ----------
    plz : Serie mit Postleitzahlen. int, float oder str; wird auf 5 Stellen
        mit führenden Nullen normalisiert.
    mapping : dict {"01067": "SN", ...}, Series (Index = PLZ), DataFrame mit
        den Spalten plz_key/subdiv_key, oder Pfad zu einer CSV mit diesen
        Spalten.
    plz_key, subdiv_key : Spaltennamen bei DataFrame/CSV.
 
    Returns
    -------
    pd.Series mit normalisierten Kürzeln, NA wo nicht auflösbar.
    """
    if isinstance(mapping, str):
        mapping = pd.read_csv(mapping, dtype=str)
    if isinstance(mapping, pd.DataFrame):
        missing = {plz_key, subdiv_key} - set(mapping.columns)
        if missing:
            raise KeyError(f"Mapping-Tabelle ohne Spalten {sorted(missing)}")
        mapping = mapping.set_index(plz_key)[subdiv_key]
    if isinstance(mapping, dict):
        mapping = pd.Series(mapping)
 
    lookup = pd.Series(
        normalize_subdiv(pd.Series(mapping.to_numpy())).to_numpy(),
        index=normalize_plz(pd.Series(mapping.index.astype(str))).to_numpy(),
    )
    lookup = lookup[~lookup.index.duplicated(keep="first")]
 
    return normalize_plz(plz).map(lookup).astype("string")
 
 
def normalize_plz(plz: pd.Series) -> pd.Series:
    """
    Normalisiert PLZ auf fünfstellige Strings mit führenden Nullen.
 
    Fängt den häufigsten Datenfehler ab: PLZ, die irgendwo als int gelesen
    wurde, verliert die führende Null - aus 01067 (Dresden) wird 1067, und die
    ostdeutschen PLZ-Bereiche 0xxxx fallen bei einem naiven String-Join
    geschlossen aus der Zuordnung. Nicht-fünfstellige Werte werden zu NA.
    """
    s = plz.astype("string").str.strip()
    # float-Artefakte wie "01067.0" bereinigen
    s = s.str.replace(r"\.0+$", "", regex=True)
    # nur reine Ziffernfolgen mit hoechstens 5 Stellen sind gueltige PLZ;
    # alles andere (Auslands-PLZ, Buchstaben, Muell) wird NA statt still zu "00000"
    s = s.where(s.str.fullmatch(r"\d{1,5}", na=False), pd.NA)
    return s.str.zfill(5)
 
 
def make_date_features(
    dates: Union[pd.Series, pd.DatetimeIndex, np.ndarray, list],
    *,
    harmonics: int = 2,
    weekly: bool = True,
    monthly: bool = True,
    holidays_mode: str = "population_share",
    subdiv: Union[pd.Series, None] = None,
    subdiv_fallback: str = "population_share",
    holiday_distance: bool = False,
    horizon: int = 14,
    weights: Union[dict[str, float], None] = None,
    prefix: str = TIME_FEATURE_PREFIX,
    index: Union[pd.Index, None] = None,
    _dedupe: bool = True,
) -> pd.DataFrame:
    """
    Baut zyklische + kalendarische Features aus einem Datum (yyyy-mm-dd).
 
    Parameters
    ----------
    dates : datum-artige Serie/Array, wird via pd.to_datetime geparst.
    harmonics : Anzahl Harmonischer der Jahresperiode (1..3 sinnvoll). k=1 ist
        eine reine Sinuswelle, k>=2 erlaubt asymmetrische Saisonalität.
    weekly, monthly : Wochen- bzw. Monatsperiode zuschalten.
    holidays_mode : Kodierung der Feiertage.
        "population_share" (Default, bundesweit): Anteil der Bevölkerung, für
            die der Tag gesetzlicher Feiertag ist. Bundeseinheitliche Feiertage
            = 1.0, Fronleichnam ~0.64, Buß- und Bettag ~0.05.
        "per_state": exakt je Bundesland, erfordert `subdiv`. Vorzuziehen,
            sobald das Bundesland (z. B. via PLZ) am Vertrag verfügbar ist.
        "nationwide_only": nur die 9 bundeseinheitlichen Feiertage, binär.
        "off": keine Feiertagsfeatures.
    subdiv : Series mit Bundesland-Angaben, nur für holidays_mode="per_state".
        Wird über normalize_subdiv() aufgelöst - Kürzel, ausgeschriebene Namen
        und Destatis-Schlüssel ("09") werden erkannt.
    subdiv_fallback : Umgang mit Zeilen ohne auflösbares Bundesland.
        "population_share" (Default): bevölkerungsgewichteter Anteil als
            beste Schätzung. "nationwide": nur bundeseinheitliche Feiertage
            (konservativ). "raise": Abbruch.
        In jedem Fall markiert TIME_FEAT_subdiv_known die betroffenen Zeilen.
    holiday_distance : zusätzlich Abstand zum letzten/nächsten Feiertag,
        auf `horizon` Tage geclippt und normiert. Default aus, um X klein zu
        halten.
    weights : überschreibt POPULATION_WEIGHTS, z. B. mit der tatsächlichen
        Bestandsverteilung {"NW": 0.31, ...}. Wird intern normiert.
    prefix : Spaltenpräfix, default TIME_FEATURE_PREFIX ("TIME_FEAT_"),
        damit die Features in der Feature-Auswahl als Block erkennbar sind.
        Siehe time_feature_columns() zum Herausfiltern.
    index : optionaler Index für das Ergebnis (default: Index der Eingabe).
 
    Returns
    -------
    pd.DataFrame, float64, garantiert ohne NaN.
    """
    if harmonics < 1:
        raise ValueError("harmonics muss >= 1 sein")
    valid_modes = {"population_share", "per_state", "nationwide_only", "off"}
    if holidays_mode not in valid_modes:
        raise ValueError(f"holidays_mode muss in {sorted(valid_modes)} liegen")
    if holidays_mode == "per_state" and subdiv is None:
        raise ValueError("holidays_mode='per_state' erfordert subdiv")
 
    d = pd.to_datetime(pd.Series(dates)).dt.normalize()
    # Zeitzone abstreifen: alles Weitere rechnet auf datetime64[D], und dabei
    # wuerde pandas tz-aware Werte nach UTC schieben - aus Mitternacht Berliner
    # Zeit wuerde der Vortag und damit der falsche Feiertag. Massgeblich ist der
    # lokale Kalendertag, so wie er in den Daten steht.
    if isinstance(d.dtype, pd.DatetimeTZDtype):
        d = d.dt.tz_localize(None)
    if index is not None:
        d.index = index
    if d.isna().any():
        raise ValueError(
            f"{int(d.isna().sum())} nicht parsebare Datumswerte (NaT). Vor dem "
            "Feature-Building explizit behandeln - NaN in X macht den CATE "
            "undefiniert."
        )
 
    # Leerer Input: Schema muss trotzdem stimmen, sonst bricht ein leerer
    # Scoring-Batch die Pipeline. Ueber einen Dummy-Tag, damit die Spalten
    # garantiert identisch zum Normalfall sind.
    if len(d) == 0:
        dummy_sub = pd.Series(["NW"]) if subdiv is not None else None
        schema = make_date_features(
            pd.Series(pd.to_datetime(["2025-01-02"])),
            harmonics=harmonics, weekly=weekly, monthly=monthly,
            holidays_mode=holidays_mode, subdiv=dummy_sub,
            subdiv_fallback=subdiv_fallback, holiday_distance=holiday_distance,
            horizon=horizon, weights=weights, prefix=prefix,
        )
        return schema.iloc[0:0].set_axis(d.index)
 
    # Deduplizierung: alle Features sind reine Funktionen von (Datum, Bundesland).
    # Bestandsdaten haben typischerweise Millionen Zeilen auf wenigen tausend
    # Stichtagen - dann lohnt sich das Rechnen auf den Unikaten deutlich.
    if _dedupe and len(d) > 5000:
        day_codes = pd.factorize(d)[0]
        if subdiv is not None:
            sub_codes, sub_uniq = pd.factorize(pd.Series(subdiv), use_na_sentinel=False)
            combo = day_codes.astype(np.int64) * max(len(sub_uniq), 1) + sub_codes
        else:
            combo = day_codes
        _, first_idx, inverse = np.unique(combo, return_index=True, return_inverse=True)
        if len(first_idx) < 0.5 * len(d):
            sub_sub = None if subdiv is None else pd.Series(subdiv).iloc[first_idx]
            uniq_feats = make_date_features(
                d.iloc[first_idx],
                harmonics=harmonics, weekly=weekly, monthly=monthly,
                holidays_mode=holidays_mode, subdiv=sub_sub,
                subdiv_fallback=subdiv_fallback, holiday_distance=holiday_distance,
                horizon=horizon, weights=weights, prefix=prefix, _dedupe=False,
            )
            return pd.DataFrame(
                uniq_feats.to_numpy()[inverse], columns=uniq_feats.columns, index=d.index
            )
 
    out: dict[str, np.ndarray] = {}
 
    # --- Jahresperiode, schaltjahrkorrekt ------------------------------------
    days_in_year = np.where(d.dt.is_leap_year, 366.0, 365.0)
    theta_y = 2.0 * np.pi * (d.dt.dayofyear.to_numpy() - 1) / days_in_year
    for k in range(1, harmonics + 1):
        suf = "" if k == 1 else str(k)
        out[f"{prefix}year_sin{suf}"] = np.sin(k * theta_y)
        out[f"{prefix}year_cos{suf}"] = np.cos(k * theta_y)
 
    # --- Wochenperiode --------------------------------------------------------
    dow = d.dt.dayofweek.to_numpy()
    if weekly:
        theta_w = 2.0 * np.pi * dow / 7.0
        out[f"{prefix}dow_sin"] = np.sin(theta_w)
        out[f"{prefix}dow_cos"] = np.cos(theta_w)
        out[f"{prefix}is_weekend"] = (dow >= 5).astype(float)
 
    # --- Monatsposition (Abbuchungs-/Zahlungszyklen) --------------------------
    if monthly:
        dom = d.dt.day.to_numpy()
        dim = d.dt.days_in_month.to_numpy()
        theta_m = 2.0 * np.pi * (dom - 1) / dim
        out[f"{prefix}dom_sin"] = np.sin(theta_m)
        out[f"{prefix}dom_cos"] = np.cos(theta_m)
        out[f"{prefix}is_month_start"] = (dom <= 3).astype(float)
        out[f"{prefix}is_month_end"] = (dim - dom <= 2).astype(float)
 
    # --- Feiertage ------------------------------------------------------------
    if holidays_mode != "off":
        for name, values in _holiday_features(
            d, dow, holidays_mode, subdiv, holiday_distance, horizon, weights,
            subdiv_fallback,
        ).items():
            out[f"{prefix}{name}"] = values
 
    df = pd.DataFrame(out, index=d.index).astype("float64")
    assert df.abs().to_numpy().max() <= 1.0 + 1e-9, "unbeschränktes Feature erzeugt"
    return df
 
 
def add_date_features(
    df: pd.DataFrame,
    date_col: str,
    subdiv_col: Union[str, None] = None,
    *,
    holidays_mode: Union[str, None] = None,
    drop_date_col: bool = False,
    overwrite: bool = False,
    warn_unresolved: bool = True,
    **kwargs,
) -> pd.DataFrame:
    """
    Hängt die Zeitfeatures an einen DataFrame an und gibt eine Kopie zurück.
 
    Der Eingabe-DataFrame wird nicht verändert. Die neuen Spalten werden
    positionsbasiert angehängt, nicht über den Index gejoint - damit
    funktioniert das auch bei doppelten oder unsortierten Index-Werten, wo ein
    join/merge still falsch alignen oder abbrechen würde.
 
    Parameters
    ----------
    df : Eingabe-DataFrame.
    date_col : Name der Datumsspalte (yyyy-mm-dd, str oder datetime).
    subdiv_col : Name der Bundesland-Spalte. Wird angegeben, schaltet die
        Funktion automatisch auf holidays_mode="per_state" und löst die Werte
        über normalize_subdiv() auf - Kürzel, ausgeschriebene Namen mit
        Titelzusatz ("Freistaat Bayern") und Destatis-Schlüssel werden erkannt.
        Ohne Angabe: bundesweiter, bevölkerungsgewichteter Feiertagsanteil.
    holidays_mode : überschreibt die Automatik, falls nötig.
    warn_unresolved : Warnung mit den nicht auflösbaren Rohwerten und ihrer
        Häufigkeit ausgeben. Diese Zeilen brechen nichts ab, sondern laufen in
        subdiv_fallback und werden über TIME_FEAT_subdiv_known markiert.
    drop_date_col : Datumsspalte im Ergebnis entfernen. Default False - das
        Rohdatum ist für Splits, Diagnostik und Zeitfenster-Auswertungen nützlich,
        gehört aber nicht in X (nicht beschränkt, extrapoliert nicht).
    overwrite : bestehende TIME_FEAT_-Spalten überschreiben statt Fehler werfen.
    **kwargs : werden an make_date_features durchgereicht (harmonics, weights,
        holidays_mode, subdiv, holiday_distance, prefix, ...).
 
    Returns
    -------
    pd.DataFrame : alle ursprünglichen Spalten plus die neuen TIME_FEAT_-Spalten.
    """
    if date_col not in df.columns:
        raise KeyError(
            f"Spalte '{date_col}' nicht im DataFrame. Vorhanden: {list(df.columns)[:20]}"
        )
 
    raw = df[date_col]
    parsed = pd.to_datetime(raw, errors="coerce")
    if parsed.isna().any() and not raw.isna().all():
        bad = raw[parsed.isna() & raw.notna()].unique()[:5]
        n_bad = int((parsed.isna()).sum())
        raise ValueError(
            f"'{date_col}': {n_bad} Werte nicht als Datum parsebar bzw. leer. "
            f"Beispiele: {list(bad)}. Vor dem Feature-Building klären - NaN in X "
            "macht den CATE undefiniert."
        )
 
    # Bundesland: Spaltenname (subdiv_col) oder direkt als Series (subdiv)
    subdiv = kwargs.pop("subdiv", None)
    if subdiv_col is not None:
        if subdiv_col not in df.columns:
            raise KeyError(
                f"Spalte '{subdiv_col}' nicht im DataFrame. "
                f"Vorhanden: {list(df.columns)[:20]}"
            )
        subdiv = df[subdiv_col]
    elif isinstance(subdiv, str):
        if subdiv not in df.columns:
            raise KeyError(f"Spalte '{subdiv}' nicht im DataFrame.")
        subdiv = df[subdiv]
 
    if subdiv is not None:
        subdiv = pd.Series(subdiv).reindex(df.index)
        resolved = normalize_subdiv(subdiv)
        if warn_unresolved and resolved.isna().any():
            offen = subdiv[resolved.isna()].astype("string").fillna("<NA>")
            top = offen.value_counts().head(10)
            warnings.warn(
                f"{len(offen)} von {len(df)} Zeilen ({len(offen) / len(df):.1%}) "
                f"ohne auflösbares Bundesland. Häufigste Rohwerte: "
                f"{dict(top)}. Diese Zeilen erhalten den Fallback und sind über "
                f"{TIME_FEATURE_PREFIX}subdiv_known markiert.",
                UserWarning,
                stacklevel=2,
            )
        kwargs["subdiv"] = resolved.reset_index(drop=True)
        holidays_mode = holidays_mode or "per_state"
 
    if holidays_mode is not None:
        kwargs["holidays_mode"] = holidays_mode
 
    feats = make_date_features(parsed.reset_index(drop=True), **kwargs)
 
    collisions = [c for c in feats.columns if c in df.columns]
    if collisions and not overwrite:
        raise ValueError(
            f"Spalten existieren bereits: {collisions}. "
            "overwrite=True setzen oder ein anderes prefix= wählen."
        )
 
    out = df.drop(columns=collisions) if collisions else df.copy()
    # Blockweise Zuweisung: positionsbasiert, kein Index-Alignment, keine
    # Fragmentierungs-Warnung durch Einzelzuweisungen.
    out[list(feats.columns)] = feats.to_numpy()
 
    if drop_date_col:
        out = out.drop(columns=[date_col])
    return out
 
 
# --------------------------------------------------------------------------- #
# Feiertagslogik
# --------------------------------------------------------------------------- #
 
def _require_holidays() -> None:
    if _holidays is None:
        raise ImportError("Paket 'holidays' nicht installiert. " + _REQUIRED_HOLIDAYS_NOTE)
 
 
@lru_cache(maxsize=256)
def _calendar(subdiv: Union[str, None], y0: int, y1: int, catholic: bool) -> tuple:
    """Sortiertes Tupel der Feiertage einer Region. Gecacht: der Aufbau des
    Kalenders dominiert sonst die Laufzeit bei zeilenweisem Scoring."""
    _require_holidays()
    categories = ("public", "catholic") if catholic else ("public",)
    cal = _holidays.country_holidays(
        "DE", subdiv=subdiv, years=range(y0, y1 + 1), categories=categories
    )
    return tuple(sorted(cal.keys()))
 
 
def _to_day_array(dates) -> np.ndarray:
    return pd.to_datetime(pd.Series(list(dates))).to_numpy().astype("datetime64[D]")
 
 
def _regions(y0: int, y1: int, weights: Union[dict, None]) -> list[tuple[float, np.ndarray]]:
    """Zerlegt Deutschland in gewichtete Teilpopulationen mit je eigenem
    Kalender. Bayern wird gesplittet, weil Mariä Himmelfahrt nur in den
    überwiegend katholischen Gemeinden gesetzlicher Feiertag ist."""
    w = dict(weights or POPULATION_WEIGHTS)
    unknown = set(w) - set(POPULATION_WEIGHTS)
    if unknown:
        raise ValueError(f"unbekannte Bundesland-Kürzel: {sorted(unknown)}")
 
    regions: list[tuple[float, np.ndarray]] = []
    for state, weight in w.items():
        if weight < 0:
            raise ValueError(f"negatives Gewicht für {state}")
        if state == "BY":
            regions.append(
                (weight * (1.0 - CATHOLIC_SHARE_BY),
                 _to_day_array(_calendar("BY", y0, y1, False)))
            )
            regions.append(
                (weight * CATHOLIC_SHARE_BY,
                 _to_day_array(_calendar("BY", y0, y1, True)))
            )
        else:
            regions.append((weight, _to_day_array(_calendar(state, y0, y1, False))))
 
    total = sum(r[0] for r in regions)
    if total <= 0:
        raise ValueError("Summe der Gewichte ist 0")
    return [(weight / total, cal) for weight, cal in regions]
 
 
def _bridge_mask(day: np.ndarray, dow: np.ndarray, cal: np.ndarray) -> np.ndarray:
    """Brückentag: Werktag, der zwischen Feiertag und Wochenende eingeklemmt ist
    (Freitag nach Donnerstags-Feiertag, Montag vor Dienstags-Feiertag)."""
    one = np.timedelta64(1, "D")
    is_hol = np.isin(day, cal)
    return (
        ~is_hol
        & (dow < 5)
        & (
            (np.isin(day - one, cal) & (dow == 4))
            | (np.isin(day + one, cal) & (dow == 0))
        )
    )
 
 
def _holiday_features(
    d: pd.Series,
    dow: np.ndarray,
    mode: str,
    subdiv: Union[pd.Series, None],
    with_distance: bool,
    horizon: int,
    weights: Union[dict, None],
    subdiv_fallback: str = "population_share",
) -> dict[str, np.ndarray]:
    _require_holidays()
    day = d.to_numpy().astype("datetime64[D]")
    y0, y1 = int(d.dt.year.min()) - 1, int(d.dt.year.max()) + 1
    n = len(d)
    known = None
 
    if mode == "population_share":
        share = np.zeros(n)
        bridge = np.zeros(n)
        for weight, cal in _regions(y0, y1, weights):
            share += weight * np.isin(day, cal)
            bridge += weight * _bridge_mask(day, dow, cal)
 
    elif mode == "per_state":
        sub = normalize_subdiv(pd.Series(subdiv).reindex(d.index))
        known = sub.notna().to_numpy()
 
        if not known.all():
            if subdiv_fallback == "raise":
                raise ValueError(
                    f"{int((~known).sum())} von {n} Zeilen ohne auflösbares "
                    "Bundesland. Mit normalize_subdiv() prüfen, welche Werte "
                    "betroffen sind, oder subdiv_fallback='population_share' "
                    "bzw. 'nationwide' setzen."
                )
            if subdiv_fallback not in {"population_share", "nationwide"}:
                raise ValueError(
                    "subdiv_fallback muss 'population_share', 'nationwide' "
                    "oder 'raise' sein"
                )
 
        share = np.zeros(n)
        bridge = np.zeros(n)
 
        for state, positions in sub.dropna().groupby(sub.dropna()).groups.items():
            pos = d.index.get_indexer(positions)
            catholic = str(state) == "BY"  # Mariä Himmelfahrt konservativ mitnehmen
            cal = _to_day_array(_calendar(str(state), y0, y1, catholic))
            share[pos] = np.isin(day[pos], cal).astype(float)
            bridge[pos] = _bridge_mask(day[pos], dow[pos], cal).astype(float)
 
        # Zeilen ohne Bundesland: bestmögliche Schätzung statt Abbruch.
        if not known.all():
            miss = ~known
            if subdiv_fallback == "population_share":
                for weight, cal in _regions(y0, y1, weights):
                    share[miss] += weight * np.isin(day[miss], cal)
                    bridge[miss] += weight * _bridge_mask(day[miss], dow[miss], cal)
            else:  # nationwide
                cal = _to_day_array(_calendar(None, y0, y1, False))
                share[miss] = np.isin(day[miss], cal).astype(float)
                bridge[miss] = _bridge_mask(day[miss], dow[miss], cal).astype(float)
 
    else:  # nationwide_only
        cal = _to_day_array(_calendar(None, y0, y1, False))
        share = np.isin(day, cal).astype(float)
        bridge = _bridge_mask(day, dow, cal).astype(float)
 
    share = np.clip(share, 0.0, 1.0)
    bridge = np.clip(bridge, 0.0, 1.0)
    res = {
        "holiday_share": share,
        "is_holiday_nationwide": (share >= 0.999).astype(float),
        "bridge_share": bridge,
        # Anteil der Bevölkerung, für die der Tag ein regulärer Arbeitstag ist.
        # Redundant für Bäume, aber die relevante Grösse für lineare Final Stages.
        "workday_share": np.where(dow >= 5, 0.0, 1.0 - share),
    }
    if mode == "per_state":
        # Stabiles Schema: Flag existiert immer im per_state-Modus, auch wenn
        # im aktuellen Batch kein Wert fehlt. Sonst weicht das Serving-Schema
        # je nach Datenqualität vom Trainings-Schema ab.
        res["subdiv_known"] = known.astype(float)
 
    if with_distance:
        nat = _to_day_array(_calendar(None, y0, y1, False))
        i = np.searchsorted(nat, day, side="left")
        nxt = np.where(i < len(nat), (nat[np.clip(i, 0, len(nat) - 1)] - day).astype(int), horizon)
        prv = np.where(i > 0, (day - nat[np.clip(i - 1, 0, len(nat) - 1)]).astype(int), horizon)
        res["days_to_holiday"] = np.clip(nxt, 0, horizon) / horizon
        res["days_since_holiday"] = np.clip(prv, 0, horizon) / horizon
 
    return res

In [18]:
df = add_date_features(df, 'AUSSPIELUNG_DT', 'BUNDESLAND')

/tmp/ipykernel_658689/1632164869.py:1: UserWarning: 15663 von 155193 Zeilen (10.1%) ohne auflösbares Bundesland. Häufigste Rohwerte: {'<NA>': np.int64(15656), ' .': np.int64(7)}. Diese Zeilen erhalten den Fallback und sind über TIME_FEAT_subdiv_known markiert.
  df = add_date_features(df, 'AUSSPIELUNG_DT', 'BUNDESLAND')


In [19]:
df

,PARTNER_ID_V,PARTNER_ID,LEAD_ID_STRING,LEAD_ART,LEAD_ART_DETAIL_TXT,LEAD_SENDEWEG,WVRM_WUNSCHVM_FLG,BASIS_DT,TA_DT_JUENGST_KDB,TA_DT_AELTEST_KDB,...,TIME_FEAT_is_weekend,TIME_FEAT_dom_sin,TIME_FEAT_dom_cos,TIME_FEAT_is_month_start,TIME_FEAT_is_month_end,TIME_FEAT_holiday_share,TIME_FEAT_is_holiday_nationwide,TIME_FEAT_bridge_share,TIME_FEAT_workday_share,TIME_FEAT_subdiv_known
0,475.0,4266.0,24JUL2026:10:17:50.762692,RS_NF,Angebotsnachfass Rechtsschutz,P,0.0,2026-07-24,NaT,NaT,...,0.0,-0.394356,0.918958,0.0,1.0,0.0,0.0,0.0,1.0,1.0
1,715.0,6520.0,22DEC2025:16:43:17.831162,RS_NF,Angebotsnachfass Rechtsschutz,E,0.0,2025-12-22,NaT,NaT,...,0.0,-0.937752,0.347305,0.0,0.0,1.0,1.0,0.0,0.0,1.0
4,1071.0,10120.0,25NOV2025:16:34:36.242665,RS_NF,Angebotsnachfass Rechtsschutz,E,0.0,2025-11-25,NaT,NaT,...,1.0,-0.406737,0.913545,0.0,1.0,0.0,0.0,0.0,0.0,1.0
6,1737.0,16986.0,13NOV2025:08:48:18.521443,HS_NF,Angebotsnachfass HUS - Privathaftpflicht,P,0.0,2025-11-13,NaT,NaT,...,0.0,-0.587785,-0.809017,0.0,0.0,0.0,0.0,0.0,1.0,1.0
8,1898.0,18808.0,04MAY2026:14:07:02.335061,HS_NF,Angebotsnachfass HUS - Privathaftpflicht,P,0.0,2026-05-04,NaT,NaT,...,1.0,0.968077,-0.250653,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
329355,152126062.0,81014245.0,31JUL2026:09:07:27.286778,HS_NF,Angebotsnachfass HUS - Privathaftpflicht,E,0.0,2026-07-31,NaT,NaT,...,0.0,0.571268,0.820763,0.0,0.0,0.0,0.0,0.0,1.0,0.0
329356,152126213.0,81015229.0,31JUL2026:10:38:00.324448,HS_NF,Angebotsnachfass HUS - HuG-Haftpflicht,E,0.0,2026-07-31,NaT,NaT,...,0.0,0.571268,0.820763,0.0,0.0,0.0,0.0,0.0,1.0,0.0
329360,152126445.0,81015511.0,31JUL2026:11:57:45.713921,HS_NF,Angebotsnachfass HUS - Wohngebäude,E,0.0,2026-07-31,NaT,NaT,...,0.0,0.571268,0.820763,0.0,0.0,0.0,0.0,0.0,1.0,0.0
329361,152126668.0,81019488.0,31JUL2026:18:27:37.260359,HS_NF,Angebotsnachfass HUS - Privathaftpflicht,E,0.0,2026-07-31,NaT,NaT,...,0.0,0.571268,0.820763,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [20]:
df.to_parquet(f'{get_current_datetime_string()}_lead_heterogeneity.parquet')

In [24]:
df.LEAD_ID_STRING

2         22DEC2025:16:43:17.831162
4         25NOV2025:16:34:36.242665
5         13NOV2025:08:48:18.521443
9         15NOV2024:10:40:36.147252
10        06FEB2025:13:06:28.436998
                    ...            
306867    02APR2026:10:55:49.065967
306870    02APR2026:13:26:41.185569
306874    02APR2026:11:31:24.168694
306878    02APR2026:15:09:39.951009
306879    02APR2026:13:13:21.595705
Name: LEAD_ID_STRING, Length: 127210, dtype: object

In [27]:
df1 = df[['PARTNER_ID_V', 'Treatment', 'TA_HUS_RS_ABSCHLUSS_CNT']].reset_index(drop=True)

In [22]:

#df2 = df2[['PARTNER_ID_V', 'Treatment', 'TA_HUS_RS_ABSCHLUSS_CNT']].reset_index(drop=True)

In [30]:
df3 = df1[['PARTNER_ID_V', 'Treatment']]
df4 = df2[['PARTNER_ID_V', 'Treatment']]

In [31]:
assert_frame_equal(df3, df4)

In [27]:
from pandas.testing import assert_frame_equal
df1 = pd.read_parquet('2026_10_29_10_52_21_lead_heterogeneity.parquet').reset_index(drop=True).drop(columns=['LEAD_ID_STRING'])
df2 = pd.read_parquet('2026_08_12_16_22_00_lead_heterogeneity.parquet').reset_index(drop=True)


assert_frame_equal(df1, df2)

AssertionError: DataFrame.iloc[:, 1] (column name="BASIS_DT") are different

DataFrame.iloc[:, 1] (column name="BASIS_DT") values are different (3.3142 %)
[index]: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, ...]
[left]:  [1766361600000, 1764028800000, 1762992000000, 1731628800000, 1738800000000, 1760313600000, 1737763200000, 1746662400000, 1771545600000, 1737072000000, 1747008000000, 1760918400000, 1757030400000, 1753228800000, 1767916800000, 1731456000000, 1734307200000, 1774224000000, 1769385600000, 1740528000000, 1751414400000, 1755648000000, 1748304000000, 1772582400000, 1732838400000, 1771804800000, 1739923200000, 1754697600000, 1752451200

In [31]:
(df1.TA_HUS_RS_ABSCHLUSS_CNT == df2.TA_HUS_RS_ABSCHLUSS_CNT)

0         True
1         True
2         True
3         True
4         True
          ... 
127205    True
127206    True
127207    True
127208    True
127209    True
Name: TA_HUS_RS_ABSCHLUSS_CNT, Length: 127210, dtype: bool